# M3 채널 확대 — 보관 모델의 4·5월 최종 평가

GPU 런타임에서 순서대로 실행합니다. 기존 학습 ZIP을 프로젝트 담당자에게 받아 내 Drive의 `QUIC_M3_results`에 보관해야 합니다. 저장소에는 가중치가 없습니다.

- 2월 검증으로 선택한 모델 4개를 고정해 4·5월 총 8회 평가합니다. 새 학습이나 재선택은 하지 않습니다.
- 원본 데이터 약 3.54GB를 다운로드하고 기존 분할의 해시를 대조합니다.
- Drive 연결은 이 노트북에서 파일에 접근할 권한을 요구합니다.
- 실제 평가 코드는 2026-11-08 실행본을 내장했습니다. 이 공개용 노트북은 출력 없이 정리한 실행 템플릿이며, 조립 후 전체 GPU 재실행은 하지 않았습니다.


In [ ]:
import sys, subprocess, os, json, hashlib, shutil
from pathlib import Path

ROOT = Path('/content/helloworld-m3-width')
COMMIT = 'a78399589391a4807df8ede9ef8205eac9d2e466'
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/RealOctavision/helloworld.git', str(ROOT)], check=True)
    if subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip() != COMMIT:
        subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', COMMIT], check=True)
        subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT, '다른 저장소 버전입니다. 별도 새 폴더를 사용하세요.'

# Install before importing NumPy/PyTorch in a fresh runtime.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy==2.1.3', 'cesnet-models==0.4.1', 'tables==3.11.1'], check=True)
os.chdir(ROOT)
import torch
assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('기존 실험 환경: A100 / Torch 2.11.0+cu130. 다른 환경은 결과에 기록됩니다.')
print('남은 디스크 GB:', round(shutil.disk_usage(ROOT).free / 1e9, 1))


In [ ]:
from google.colab import drive
import zipfile

drive.mount('/content/drive')
EXP = ROOT/'experiments/2026-11-05_m3_width'
# Team members need the original training ZIP from the project artifact owner.
ARCHIVE = Path('/content/drive/MyDrive/QUIC_M3_results/m3_width_results_20261105T060434Z.zip')
def sha(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
assert sha(ARCHIVE) == '6b6bb9e8f265aab96e866f53f31152bc11c1f910b02322648964b17bc263653b'
with zipfile.ZipFile(ARCHIVE) as z:
    assert z.testzip() is None
    for item in z.infolist():
        target = (ROOT/item.filename).resolve()
        assert target.is_relative_to(EXP.resolve()), item.filename
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        payload = z.read(item)
        if target.exists():
            assert target.read_bytes() == payload, f'Existing file differs: {target}'
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(payload)
print('Four completed fits restored; ZIP SHA-256 and CRC passed; no training.')


In [ ]:
SOURCE = ROOT / 'research/2026-09-22_plan_support/data/CESNET-QUICEXT-25-XS.h5'
SOURCE.parent.mkdir(parents=True, exist_ok=True)
SIZE = 3544625537
EXPECTED_SHA = '6b323c4c53b7c9d0ec9011f895b3c5ba6b35247576e41ef19fe84ab8632810ab'
def sha(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
if not SOURCE.exists() or SOURCE.stat().st_size != SIZE:
    subprocess.run(['curl', '-fL', '--retry', '3', '-C', '-',
        'https://liberouter.org/datazoo/download?bucket=cesnet-quicext-25&file=CESNET-QUICEXT-25-XS.h5',
        '-o', str(SOURCE)], check=True)
assert SOURCE.stat().st_size == SIZE and sha(SOURCE) == EXPECTED_SHA, '데이터 크기/해시 불일치: 학습하지 말고 다운로드를 점검하세요.'
print('공식 데이터 크기·SHA-256 일치')


In [ ]:
BASE = ROOT / 'experiments/2026-10-30_midterm_restart'
AUDIT = ROOT / 'experiments/2026-10-28_data_audit'
BACKUP = ROOT / 'tmp/m3_width_published'
if not BACKUP.exists():
    assert not (AUDIT/'ppi.npy').exists() and not any((BASE/'prepared').rglob('*.npz')), '이미 실제 데이터가 있습니다. 기존 재현 환경을 임의 초기화하지 마세요.'
    BACKUP.mkdir(parents=True)
# Each move is independently resumable if the notebook stopped during setup.
if not (BACKUP/'audit_manifest.json').exists():
    shutil.move(str(AUDIT/'manifest.json'), str(BACKUP/'audit_manifest.json'))
if not (BACKUP/'prepared').exists():
    shutil.move(str(BASE/'prepared'), str(BACKUP/'prepared'))

subprocess.run([sys.executable, str(AUDIT/'analyze.py'), '--prepare-only'], check=True)
subprocess.run([sys.executable, str(BASE/'data.py'), 'self-test'], check=True)
subprocess.run([sys.executable, str(BASE/'data.py'), 'prepare'], check=True)

old = json.loads((BACKUP/'prepared/manifest.json').read_text())
new = json.loads((BASE/'prepared/manifest.json').read_text())
for key in ['config_sha256', 'source_sha256', 'cache_sha256', 'cohorts']:
    assert old[key] == new[key], f'기존 데이터와 불일치: {key}'
for rel, expected in old['artifacts'].items():
    if rel.endswith('.npz'):
        assert sha(BASE/'prepared'/rel) == expected, f'준비 배열 불일치: {rel}'
print('기존 데이터 캐시·분할·NPZ 해시 모두 일치')


In [ ]:
EVALUATOR_SOURCE = '"""Evaluate the four preserved February-selected checkpoints; never fit or tune."""\nimport os\nos.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\nimport argparse\nimport json\nimport platform\nimport time\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nimport numpy as np\nimport torch\nimport run as wide\nimport colab_continuation as evaluator\nfrom colab_main import compact_metrics\nfrom data import digest, read_split, write_json\nfrom training import hash_bytes, load_checkpoint, logits_for\n\nHERE = Path(__file__).resolve().parent\nOUTPUT = HERE/\'final_evaluation\'\nMONTHS = (\'202504\', \'202505\')\nCASES = ((200, 201), (200, 202), (1000, 201), (1000, 202))\nEVALUATOR_HASHES = {\n    \'colab_main.py\': \'fd7d7b50f5ba9dcfcf577fda861a267c22426673aa6a31d13467b30567489e4a\',\n    \'colab_continuation.py\': \'9cb0eabde55cb7bfa64d5649e70ce2c85f996625214c460c8f419a8dc117bf36\',\n}\n\n\ndef factory(kind, classes, seed, cfg, load_pretrained=False):\n    if load_pretrained:\n        raise ValueError(\'No pretrained teacher in this evaluation\')\n    return wide.make_wide_model(kind, classes, seed, cfg)\n\n\ndef masks_for(data, groups, budget, seed):\n    if not np.array_equal(data[\'id\'], groups[\'id\']):\n        raise ValueError(\'Evaluation/group IDs differ\')\n    overlap = groups[f\'seed{seed}_n{budget}_train_overlap\']\n    unseen = groups[\'common_unseen\']\n    if overlap.shape != data[\'y\'].shape or unseen.shape != overlap.shape or np.any(overlap & unseen):\n        raise ValueError(\'Invalid evaluation grouping\')\n    return dict(all=np.ones(len(overlap), bool), train_overlap=overlap,\n                train_nonoverlap=~overlap, common_unseen=unseen)\n\n\ndef verify_prediction_file(folder, data, groups, budget, seed):\n    record = json.loads((folder/\'metrics.json\').read_text())\n    if digest(folder/\'predictions.npz\') != record[\'prediction_sha256\']:\n        raise ValueError(\'Saved predictions changed\')\n    with np.load(folder/\'predictions.npz\', allow_pickle=False) as z:\n        if not np.array_equal(z[\'id\'], data[\'id\']) or not np.array_equal(z[\'y\'], data[\'y\']):\n            raise ValueError(\'Saved prediction IDs/labels differ\')\n        if not np.isfinite(z[\'true_log_probability\']).all():\n            raise ValueError(\'Nonfinite saved predictions\')\n        for group, mask in masks_for(data, groups, budget, seed).items():\n            score = compact_metrics(z[\'y\'][mask], z[\'predicted_label\'][mask], z[\'true_log_probability\'][mask], 20)\n            if score != record[\'groups\'][group]:\n                raise ValueError(\'Stored metrics do not match predictions\')\n    return record\n\n\ndef published_records():\n    records = {}\n    for budget, directory in [(200, wide.BASE/\'main_recovered\'),\n                              (1000, wide.BASE/\'continuation_recovered/mixed_n1000\')]:\n        paths = list(directory.glob(\'evaluation_*/verified_evaluation.json\'))\n        if len(paths) != 1:\n            raise ValueError(\'Ambiguous or missing baseline evaluation\')\n        verified = json.loads(paths[0].read_text())\n        if verified[\'verified\'] is not True:\n            raise ValueError(\'Baseline evaluation is not verified\')\n        for record in verified[\'records\']:\n            if record[\'model\'] == \'M3\' and (budget, record[\'seed\']) in CASES:\n                key = (record[\'month\'], budget, record[\'seed\'])\n                if key in records or record[\'budget\'] != budget:\n                    raise ValueError(\'Duplicate/mismatched baseline\')\n                records[key] = record\n    if set(records) != {(m, b, s) for m in MONTHS for b, s in CASES}:\n        raise ValueError(\'Missing paired baselines\')\n    return records\n\n\ndef summarize(rows):\n    result = []\n    for month in MONTHS:\n        for budget in (200, 1000):\n            for group in (\'all\', \'common_unseen\'):\n                selected = [r for r in rows if (r[\'month\'], r[\'budget\'], r[\'group\']) == (month, budget, group)]\n                if len(selected) != 2:\n                    continue\n                result.append(dict(month=month, budget=budget, group=group, seeds=[201, 202], n=selected[0][\'n\'],\n                    **{key: float(np.mean([r[key] for r in selected])) for key in (\n                        \'wide_macro_f1\', \'baseline_macro_f1\', \'delta_f1_pp\',\n                        \'wide_accuracy\', \'baseline_accuracy\', \'delta_accuracy_pp\')}))\n    return result\n\n\ndef run_evaluation():\n    cfg, frozen = wide.settings()\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required for the preserved evaluation routine\')\n    for name, expected in EVALUATOR_HASHES.items():\n        if digest(wide.BASE/name) != expected:\n            raise ValueError(\'Original evaluator changed: \'+name)\n    torch.set_num_threads(2)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    old = published_records()\n    val = read_split(wide.COHORT, \'validation\')\n    checkpoints = {}\n    for budget, seed in CASES:\n        folder = HERE/\'runs\'/f\'M3_wide_s{seed}_n{budget}\'\n        metric = wide.completed_result(folder)\n        context = json.loads((folder/\'run_context.json\').read_text())\n        if metric is None or metric[\'completed_epochs\'] != 50 or context[\'source_hashes\'][\'runner\'] != digest(HERE/\'run.py\'):\n            raise ValueError(\'Incomplete or changed training run\')\n        hashes = wide.split_hashes(read_split(wide.COHORT, f\'main_seed{seed}\', budget), val)\n        if any(context[k] != v for k, v in hashes.items()):\n            raise ValueError(\'Training or validation data changed\')\n        paired = wide.baseline(seed, budget, hashes)\n        original_training = json.loads((wide.BASE/paired[\'path\']).read_text())\n        for month in MONTHS:\n            if old[month, budget, seed][\'checkpoint_sha256\'] != original_training[\'best_checkpoint_sha256\']:\n                raise ValueError(\'Baseline test record belongs to a different checkpoint\')\n        cp = folder/\'best.pt\'\n        model = factory(wide.KIND, 20, seed, cfg).to(\'cuda\')\n        payload = load_checkpoint(cp, model)\n        if payload[\'epoch\'] != metric[\'selected_epoch\']:\n            raise ValueError(\'Checkpoint is not the February-selected epoch\')\n        repeated = logits_for(model, val[\'X\'], torch.device(\'cuda\')).softmax(1).numpy()\n        with np.load(folder/\'validation_predictions.npz\', allow_pickle=False) as z:\n            np.testing.assert_array_equal(z[\'id\'], val[\'id\'])\n            np.testing.assert_array_equal(z[\'y\'], val[\'y\'])\n            np.testing.assert_allclose(repeated, z[\'probabilities\'], atol=1e-6, rtol=1e-5)\n            np.testing.assert_array_equal(repeated.argmax(1), z[\'probabilities\'].argmax(1))\n        checkpoints[f\'M3_wide_s{seed}_n{budget}\'] = dict(sha256=digest(cp), selected_epoch=payload[\'epoch\'])\n        del model\n    protocol = dict(checkpoints=checkpoints, months=list(MONTHS), cases=list(map(list, CASES)),\n        selection=\'Preserved February-selected best.pt; no retraining or post-test selection\',\n        primary=\'April all\', temporal=\'May all\', secondary=\'common_unseen and overlap groups\',\n        code_sha256=digest(Path(__file__)), evaluator_hashes=EVALUATOR_HASHES,\n        python=platform.python_version(), torch=str(torch.__version__), numpy=np.__version__,\n        cuda=torch.version.cuda, device=torch.cuda.get_device_name(0), precision=\'FP32, TF32 disabled\',\n        batch_size=frozen[\'evaluation\'][\'batch_size\'])\n    wide.bind_context(OUTPUT, protocol)\n    protocol_sha = digest(OUTPUT/\'run_context.json\')\n    rows = []\n    started = time.monotonic()\n    def check_budget():\n        if time.monotonic()-started > 20*60:\n            raise TimeoutError(\'Evaluation time limit; completed files are reusable\')\n    for month in MONTHS:\n        data = read_split(wide.COHORT, f\'evaluation_{month}\')\n        groups = read_split(wide.COHORT, f\'evaluation_groups_{month}\')\n        hashes = {f\'evaluation_{field}_sha256\': hash_bytes(data[key])\n                  for field, key in [(\'ids\', \'id\'), (\'input\', \'X\'), (\'labels\', \'y\')]}\n        for budget, seed in CASES:\n            name = f\'M3_wide_s{seed}_n{budget}\'\n            cp = HERE/\'runs\'/name/\'best.pt\'\n            baseline = old[month, budget, seed]\n            if baseline[\'classes\'] != cfg[\'cohorts\'][wide.COHORT][\'classes\'] or any(baseline[k] != v for k, v in hashes.items()):\n                raise ValueError(\'Baseline test inputs/labels/order differ\')\n            if digest(cp) != checkpoints[name][\'sha256\']:\n                raise ValueError(\'Checkpoint changed after freeze\')\n            masks = masks_for(data, groups, budget, seed)\n            for group, mask in masks.items():\n                if int(mask.sum()) != baseline[\'groups\'][group][\'n\']:\n                    raise ValueError(\'Baseline test group differs\')\n            folder = OUTPUT/month/name\n            with patch.object(evaluator, \'make_model\', factory):\n                evaluator.eval_one(wide.KIND, seed, budget, wide.COHORT, cp, folder,\n                    cfg, frozen, protocol_sha, data, groups, None, check_budget)\n            result = verify_prediction_file(folder, data, groups, budget, seed)\n            for group, score in result[\'groups\'].items():\n                base_score = baseline[\'groups\'][group]\n                if score[\'support\'] != base_score[\'support\']:\n                    raise ValueError(\'Baseline class support differs\')\n                rows.append(dict(month=month, budget=budget, seed=seed, group=group, n=score[\'n\'],\n                    wide_macro_f1=score[\'macro_f1\'], baseline_macro_f1=base_score[\'macro_f1\'],\n                    wide_accuracy=score[\'accuracy\'], baseline_accuracy=base_score[\'accuracy\'],\n                    delta_f1_pp=None if score[\'n\']==0 else 100*(score[\'macro_f1\']-base_score[\'macro_f1\']),\n                    delta_accuracy_pp=None if score[\'n\']==0 else 100*(score[\'accuracy\']-base_score[\'accuracy\'])))\n            print(\'EVALUATED\', month, name, json.dumps(rows[-4]), flush=True)\n            write_json(OUTPUT/\'summary.json\', dict(status=\'running\', completed=len(rows)//4, planned=8,\n                rows=rows, means=summarize(rows), protocol_sha256=protocol_sha))\n        del data, groups\n    summary = dict(status=\'completed\', completed=8, planned=8, rows=rows, means=summarize(rows),\n        protocol_sha256=protocol_sha, seconds=time.monotonic()-started,\n        notes=[\'Two paired seeds, not five-seed baseline means\', \'Baseline trained/evaluated on A100; wide on T4\',\n               \'No test scores used to choose weights; future tuning would make these reused test sets\'])\n    write_json(OUTPUT/\'summary.json\', summary)\n    print(json.dumps(dict(status=summary[\'status\'], completed=8, means=summary[\'means\']), indent=2), flush=True)\n\n\ndef self_test():\n    import tempfile\n    cfg, frozen = wide.settings()\n    x = np.zeros((40, 3, 30), dtype=\'i4\')\n    x[:, 1, :10] = 1\n    x[:, 2, :10] = np.arange(40)[:, None]+100\n    data = dict(id=np.arange(40), X=x, y=np.arange(40)%20)\n    groups = dict(id=data[\'id\'].copy(), seed201_n200_train_overlap=np.arange(40)<4, common_unseen=np.arange(40)>=8)\n    with tempfile.TemporaryDirectory() as temp:\n        cp = Path(temp)/\'best.pt\'\n        model = factory(wide.KIND, 20, 201, cfg).to(\'cuda\')\n        torch.save(dict(model=model.state_dict(), epoch=1), cp)\n        out = Path(temp)/\'202504\'/\'M3_wide_s201_n200\'\n        with patch.object(evaluator, \'make_model\', factory):\n            evaluator.eval_one(wide.KIND, 201, 200, wide.COHORT, cp, out, cfg, frozen, \'synthetic\', data, groups, None, lambda: None)\n        record = verify_prediction_file(out, data, groups, 200, 201)\n        probability = logits_for(model, x, torch.device(\'cuda\')).softmax(1).numpy()\n        expected = wide.training.metrics(data[\'y\'], probability, 20)\n        assert record[\'groups\'][\'all\'][\'macro_f1\'] == expected[\'macro_f1\']\n        assert record[\'groups\'][\'all\'][\'accuracy\'] == expected[\'accuracy\']\n        with patch.object(evaluator, \'make_model\', side_effect=AssertionError(\'Unexpected rerun\')):\n            evaluator.eval_one(wide.KIND, 201, 200, wide.COHORT, cp, out, cfg, frozen, \'synthetic\', data, groups, None, lambda: None)\n        for bad_groups in [dict(groups, id=data[\'id\'][::-1]), dict(groups, common_unseen=np.ones(40, bool))]:\n            try:\n                masks_for(data, bad_groups, 200, 201)\n            except ValueError:\n                pass\n            else:\n                raise AssertionError(\'Bad grouping accepted\')\n        (out/\'predictions.npz\').write_bytes(b\'corrupt\')\n        try:\n            verify_prediction_file(out, data, groups, 200, 201)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(\'Corrupt predictions accepted\')\n    print(\'PASS: synthetic inference, metric replay, completed reuse, ID/overlap and corruption guards; no fitting\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--self-test\', action=\'store_true\')\n    args = parser.parse_args()\n    self_test() if args.self_test else run_evaluation()\n'
script = EXP/'final_evaluate.py'
if script.exists():
    assert script.read_text() == EVALUATOR_SOURCE, 'Existing evaluator differs'
else:
    script.write_text(EVALUATOR_SOURCE)
assert sha(script) == '0cb8615404d22bfaa99f5dc1dfac4b7b29aea3390d1f6e7f00015045106c2988'


In [ ]:
for suffix, args in [('self_test', ['--self-test']), ('run', [])]:
    log = EXP/f'final_evaluation_{suffix}.log'
    with log.open('w') as f:
        result = subprocess.run([sys.executable, str(script), *args], stdout=f, stderr=subprocess.STDOUT)
    print(log.read_text()[-15000:])
    result.check_returncode()
summary = json.loads((EXP/'final_evaluation/summary.json').read_text())
assert summary['status'] == 'completed' and summary['completed'] == 8
print(json.dumps(summary['means'], indent=2))


In [ ]:
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
archive = Path('/content')/f'm3_width_final_evaluation_{stamp}.zip'
paths = [script, EXP/'final_evaluation_self_test.log', EXP/'final_evaluation_run.log']
paths += [p for p in (EXP/'final_evaluation').rglob('*') if p.is_file()]
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, p.relative_to(ROOT))
    for name in ['config.json','main_frozen.json','data.py','models.py','training.py','colab_main.py','colab_continuation.py']:
        p = ROOT/'experiments/2026-10-30_midterm_restart'/name
        z.write(p, p.relative_to(ROOT))
folder = ARCHIVE.parent
target = folder/archive.name
assert not target.exists(), 'Backup name already exists'
shutil.copy2(archive, target)
assert sha(target) == sha(archive)
with zipfile.ZipFile(target) as z:
    assert z.testzip() is None
summary_target = folder/f'm3_width_final_evaluation_{stamp}_summary.json'
shutil.copy2(EXP/'final_evaluation/summary.json', summary_target)
assert sha(summary_target) == sha(EXP/'final_evaluation/summary.json')
receipt = dict(status='completed_and_saved',evaluations=8,archive_name=target.name,bytes=target.stat().st_size,sha256=sha(target),zip_crc=True)
(folder/f'm3_width_final_evaluation_{stamp}_receipt.json').write_text(json.dumps(receipt,indent=2))
print(json.dumps(receipt,indent=2))
print('Drive backup verified. Checkpoint files remain in the original training ZIP.')
